In [2]:
%load_ext autoreload
%autoreload 2

from datetime import date, datetime, timedelta
from functools import partial
from typing import Dict, Optional

import numpy as np
import polars as pl
from tqdm import tqdm
import time

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, assign_forwards
from okx.recipes.options import prepare_options
from evaluation.forwards_eval import evaluate_parity, summarize_parity, evaluate_pillar_fit, evaluate_loeo

from okx.features import generate_time_bins

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

In [4]:
# Shared parameters
inst_family = 'BTC-USD'

# Scenario (a): Full month with binning
def make_dates(n):
    return [date(2025, 9, 1) + timedelta(days=i) for i in range(n)]
dates_month = make_dates(30)
binning_month = '5m'

# Scenario (b): Single day with option timestamps
dates_day = [date(2025, 9, 2)]
binning_day = None  # Will use unique_times from OPTIONS

In [7]:
df_futures_1 = store.get(
    inst_family=inst_family,
    inst_type='FUTURES',
    dates=make_dates(10),
    depth=0,
    binning='5m',
    features=['trim', 'strip', 'bin_ff', 'sink_bins'],
    verbose=True,
    benchmark=True,
    batch_days=5
).collect()

[store] Getting BTC-USD/FUTURES for 10 dates (depth=0, 5m binning, 4 features)
Applying sink bins after: False


Processing batches:   0%|          | 0/2 [00:00<?, ?it/s]

  - applied 'trim'
  [benchmark] trim: 0.333s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'bin_ff'
  [benchmark] bin_ff: 2.894s
  - applied 'sink_bins'
  [benchmark] sink_bins: 0.000s
  - applied 'trim'
  [benchmark] trim: 0.183s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'bin_ff'
  [benchmark] bin_ff: 2.057s
  - applied 'sink_bins'
  [benchmark] sink_bins: 0.000s
  [benchmark] Direct build: 5.479s
Using forward fill: True
Batching: True
  [benchmark] Deduplication: 0.000s


In [8]:
df_futures_2 = store.get(
    inst_family=inst_family,
    inst_type='FUTURES',
    dates=make_dates(10),
    depth=0,
    binning='5m',
    features=['trim', 'strip', 'bin_ff', 'sink_bins_after'],
    verbose=True,
    benchmark=True,
    batch_days=5
).collect()

[store] Getting BTC-USD/FUTURES for 10 dates (depth=0, 5m binning, 4 features)
Applying sink bins after: True


Processing batches:   0%|          | 0/2 [00:00<?, ?it/s]

  - applied 'trim'
  [benchmark] trim: 0.228s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'bin_ff'
  [benchmark] bin_ff: 2.802s
  - applied 'trim'
  [benchmark] trim: 0.166s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'bin_ff'
  [benchmark] bin_ff: 2.587s
  [benchmark] Direct build: 5.792s
Using forward fill: True
Batching: True
  [benchmark] Deduplication: 0.000s
  [benchmark] Sink bins: 0.000s


In [18]:
is_equal = df_futures_1.equals(df_futures_2)
print(f"is_equal: {is_equal}")
def df_is_equal(df1: pl.DataFrame, df2: pl.DataFrame, col: str) -> bool:
    return df1.select(col).equals(df2.select(col))
print(f"timeMs is equal: {df_is_equal(df_futures_1, df_futures_2, 'timeMs')}")
print(f"symbol is equal: {df_is_equal(df_futures_1, df_futures_2, 'symbol')}")
def df_is_sorted(df: pl.DataFrame, col: str ) -> bool:
    return df.select(col).equals(df.select(col).sort(col))
print(f"df1 timeMs is sorted: {df_is_sorted(df_futures_1, 'timeMs')}")
print(f"df2 timeMs is sorted: {df_is_sorted(df_futures_2, 'timeMs')}")
print(f"df1 symbol is sorted: {df_is_sorted(df_futures_1, 'symbol')}")
print(f"df2 symbol is sorted: {df_is_sorted(df_futures_2, 'symbol')}")


is_equal: False
timeMs is equal: False
symbol is equal: True
df1 timeMs is sorted: False
df2 timeMs is sorted: False
df1 symbol is sorted: False
df2 symbol is sorted: False


In [47]:
t0 = time.perf_counter()
lf_futures = lf_futures.sort(['timeMs'])
df_futures = lf_futures.collect()
print(time.perf_counter() - t0)
df_futures.head()

0.004502541967667639


symbol,timeMs,mid,time_bin
str,i64,f64,i64
"""BTC-USD-251031.OK""",1756685099757,109271.2,1756685100000
"""BTC-USD-251226.OK""",1756685099928,110518.05,1756685100000
"""BTC-USD-260327.OK""",1756685099928,112600.65,1756685100000
"""BTC-USD-250905.OK""",1756685099938,108146.05,1756685100000
"""BTC-USD-260626.OK""",1756685099958,114669.65,1756685100000


In [35]:
df_futures = lf_futures.collect()
is_sorted = df_futures.select('timeMs').equals(df_futures.select('timeMs').sort('timeMs'))
print(is_sorted)
def sort_by_ms(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.sort('timeMs')
def is_sorted_by_ms(lf: pl.LazyFrame, inst_family: str, inst_type: str, date_str: str) -> bool:
    df = lf.collect()
    is_sorted = df.select('timeMs').equals(df.select('timeMs').sort('timeMs'))
    print(f'{inst_family}/{inst_type}/{date_str} is sorted: {is_sorted}')
def is_sorted_by_exch_time(lf: pl.LazyFrame, inst_family: str, inst_type: str, date_str: str) -> bool:
    df = lf.collect()
    is_sorted = df.select('exchTimeMs').equals(df.select('exchTimeMs').sort('exchTimeMs'))
    print(f'{inst_family}/{inst_type}/{date_str} is sorted by exchTime: {is_sorted}')



False


In [36]:
store.inspect(is_sorted_by_ms, variant='raw', verbose=True)

[1/246] Inspecting BTC-USD/SWAP/2025-09-06
BTC-USD/SWAP/2025-09-06 is sorted: True
[2/246] Inspecting BTC-USD/SWAP/2025-09-07
BTC-USD/SWAP/2025-09-07 is sorted: True
[3/246] Inspecting BTC-USD/SWAP/2025-09-08
BTC-USD/SWAP/2025-09-08 is sorted: True
[4/246] Inspecting BTC-USD/SWAP/2025-09-05
BTC-USD/SWAP/2025-09-05 is sorted: True
[5/246] Inspecting BTC-USD/SWAP/2025-09-03
BTC-USD/SWAP/2025-09-03 is sorted: True
[6/246] Inspecting BTC-USD/SWAP/2025-09-04
BTC-USD/SWAP/2025-09-04 is sorted: True
[7/246] Inspecting BTC-USD/SWAP/2025-09-01
BTC-USD/SWAP/2025-09-01 is sorted: True
[8/246] Inspecting BTC-USD/SWAP/2025-09-02
BTC-USD/SWAP/2025-09-02 is sorted: True
[9/246] Inspecting BTC-USD/SWAP/2025-09-09
BTC-USD/SWAP/2025-09-09 is sorted: True
[10/246] Inspecting BTC-USD/SWAP/2025-09-10
BTC-USD/SWAP/2025-09-10 is sorted: True
[11/246] Inspecting BTC-USD/SWAP/2025-09-14
BTC-USD/SWAP/2025-09-14 is sorted: True
[12/246] Inspecting BTC-USD/SWAP/2025-09-11
BTC-USD/SWAP/2025-09-11 is sorted: True
[

KeyboardInterrupt: 

In [37]:
store.migrate(sort_by_ms, variant='raw', verbose=True)

[1/246] Migrating BTC-USD/SWAP/2025-09-06
[2/246] Migrating BTC-USD/SWAP/2025-09-07
[3/246] Migrating BTC-USD/SWAP/2025-09-08
[4/246] Migrating BTC-USD/SWAP/2025-09-05
[5/246] Migrating BTC-USD/SWAP/2025-09-03
[6/246] Migrating BTC-USD/SWAP/2025-09-04
[7/246] Migrating BTC-USD/SWAP/2025-09-01
[8/246] Migrating BTC-USD/SWAP/2025-09-02
[9/246] Migrating BTC-USD/SWAP/2025-09-09
[10/246] Migrating BTC-USD/SWAP/2025-09-10
[11/246] Migrating BTC-USD/SWAP/2025-09-14
[12/246] Migrating BTC-USD/SWAP/2025-09-11
[13/246] Migrating BTC-USD/SWAP/2025-09-13
[14/246] Migrating BTC-USD/SWAP/2025-09-16
[15/246] Migrating BTC-USD/SWAP/2025-09-12
[16/246] Migrating BTC-USD/SWAP/2025-09-18
[17/246] Migrating BTC-USD/SWAP/2025-09-17
[18/246] Migrating BTC-USD/SWAP/2025-09-15
[19/246] Migrating BTC-USD/SWAP/2025-09-20
[20/246] Migrating BTC-USD/SWAP/2025-09-21
[21/246] Migrating BTC-USD/SWAP/2025-09-19
[22/246] Migrating BTC-USD/SWAP/2025-09-24
[23/246] Migrating BTC-USD/SWAP/2025-09-27
[24/246] Migrating B

In [21]:
t0 = time.perf_counter()
time_range = lf_futures.select([
    pl.col('timeMs').min().alias('min_time'),
    pl.col('timeMs').max().alias('max_time')
]).collect()
start_ms = time_range['min_time'][0]
end_ms = time_range['max_time'][0]
print(time.perf_counter() - t0)
print(start_ms, end_ms)
from datetime import datetime, timezone

start_dt = datetime.fromtimestamp(start_ms / 1000, tz=timezone.utc)
end_dt = datetime.fromtimestamp(end_ms / 1000, tz=timezone.utc)
print("Start datetime:", start_dt)
print("End datetime:", end_dt)


0.0066939579555764794
1756771200008 1756943999993
Start datetime: 2025-09-02 00:00:00.008000+00:00
End datetime: 2025-09-03 23:59:59.993000+00:00
